# **PolypGen Challenge**

* Leo Rebecca
* Liendo Maria Sol
* Scarcelli Erica
* Venerito Silvia


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip uninstall torch torchvision torchaudio -y
!pip cache purge

!pip install torch==2.1.0+cu121 --index-url https://download.pytorch.org/whl/cu121
!pip install torchvision==0.16.0+cu121 --no-deps --index-url https://download.pytorch.org/whl/cu121
!pip install torchaudio==2.1.0+cu121 --no-deps --index-url https://download.pytorch.org/whl/cu121

In [ ]:
!pip install mmcv==2.1.0 -f https://download.openmmlab.com/mmcv/dist/cu121/torch2.1/index.html

In [ ]:
!pip install mmsegmentation==1.2.2
!pip install mmengine==0.10.3
!pip install ftfy==6.2.3

In [ ]:
!pip uninstall -y transformers safetensors

In [ ]:
!pip uninstall -y numpy
!pip install numpy==1.26.4 --force-reinstall
# Attendere il termine dell'installazione poi Runtime -> Restart Session

In [ ]:
# Check Pytorch installation
import torch, torchvision
print(torch.__version__, torch.cuda.is_available())

# Check MMSegmentation installation
import mmseg
print(mmseg.__version__)

2.1.0+cu121 True
1.2.2


In [ ]:
# Librerie di base per la manipolazione di file e dati
import os, gc
import sys
import shutil
import random
from pathlib import Path
from tqdm.auto import tqdm
from tqdm import tqdm
import numpy as np
import pandas as pd
from PIL import Image

# Librerie per la visualizzazione e l'analisi di immagini
import cv2
from google.colab.patches import cv2_imshow
import matplotlib.pyplot as plt
from skimage import color, io as img_sk, transform
from scipy import ndimage
from scipy.ndimage import convolve, binary_opening, binary_fill_holes
import skimage
from skimage import io
from glob import glob
from scipy.spatial.distance import euclidean

# Librerie per machine learning e preparazione dati
from sklearn.model_selection import train_test_split

# Librerie specifiche di MMSegmentation (per il modello)
import mmcv
from mmengine.config import Config
from mmengine.logging import print_log
from mmengine.runner import Runner
from mmseg.registry import DATASETS, RUNNERS
from mmseg.apis import init_model, inference_model
from mmseg.datasets import BaseSegDataset

/usr/local/lib/python3.11/dist-packages/albumentations/__init__.py:24: UserWarning: A new version of Albumentations is available: 2.0.2 (you have 1.4.20). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()


# DATASET

In [ ]:
group_name = 'FMZ13'

PATHs

In [ ]:
'''CARTELLA PRINCIPALE'''

working_folder = f'/content/drive/MyDrive/PolypGen/{group_name}'

checkpoint_path = f'{working_folder}/checkpoint.pth'
# per essempio best_mDice_iter_5530.pth nella cartella di path_weigths
cfg_file = f'{working_folder}/config.py'
# cfg_file = os.path.join(WEIGHTS,"unet-s5-d16_fcn_4xb4-160k_cityscapes-512x1024.py")
# path_weights = '/content/drive/MyDrive/Colab Notebooks/PolypGen-EIM-24-25/weights_unet/addestramento_preprocessing'  # cercarlo manualmente, fare una coppia e rinnombrarlo config.py

'''CARTELLA RISULTATI'''

results_folder  = f'{working_folder}/results_test'
if not os.path.exists(results_folder):
    os.makedirs(results_folder)

'''CARICAMENTO SEQUENZA TEST'''

test_img_folder = '/content/drive/MyDrive/Colab Notebooks/PolypGen-EIM-24-25/sequenceData/seq2'


In [ ]:
''' CONFIGURAZIONE MODELLO '''

cfg = Config.fromfile(cfg_file)
cfg.device='cuda'

# Visualizzazione del config caricato
print(f'Config:\n{cfg.pretty_text}')

''' INIZIALIZZAZIONE MODELLO '''

# Initialize the model from the config and the checkpoint
model = init_model(cfg, checkpoint_path)

Config:
crop_size = (
    256,
    256,
)
data_preprocessor = dict(
    bgr_to_rgb=True,
    mean=[
        0,
        0,
        0,
    ],
    pad_val=0,
    seg_pad_val=255,
    size=(
        256,
        256,
    ),
    std=[
        1,
        1,
        1,
    ],
    type='SegDataPreProcessor')
data_root = '/content/drive/MyDrive/Lab_EIM/Progetto_finale_EIM/Divisione_dataset5'
dataset_type = 'PolypGen'
default_hooks = dict(
    checkpoint=dict(
        by_epoch=False,
        interval=448,
        max_keep_ckpts=1,
        rule='greater',
        save_best='mDice',
        type='CheckpointHook'),
    early_stopping=dict(
        min_delta=0.01,
        monitor='mIoU',
        patience=10,
        rule='greater',
        type='EarlyStoppingHook'),
    logger=dict(interval=50, log_metric_by_epoch=False, type='LoggerHook'),
    param_scheduler=dict(type='ParamSchedulerHook'),
    sampler_seed=dict(type='DistSamplerSeedHook'),
    timer=dict(type='IterTimerHook'),
    visualization=d

/usr/local/lib/python3.11/dist-packages/mmseg/models/decode_heads/decode_head.py:120: UserWarning: For binary segmentation, we suggest using`out_channels = 1` to define the outputchannels of segmentor, and use `threshold`to convert `seg_logits` into a predictionapplying a threshold
  warnings.warn('For binary segmentation, we suggest using'
/usr/local/lib/python3.11/dist-packages/mmseg/models/builder.py:36: UserWarning: ``build_loss`` would be deprecated soon, please use ``mmseg.registry.MODELS.build()`` 
  warnings.warn('``build_loss`` would be deprecated soon, please use '
/usr/local/lib/python3.11/dist-packages/mmseg/models/losses/cross_entropy_loss.py:250: UserWarning: Default ``avg_non_ignore`` is False, if you would like to ignore the certain label and average loss over non-ignore labels, which is the same with PyTorch official cross_entropy, set ``avg_non_ignore=True``.
  warnings.warn(


Loads checkpoint by local backend from path: /content/drive/MyDrive/PolypGen/Pepine/checkpoint.pth


# Preprocessing TESTING

In [ ]:
# le stesse funzioni del Training

In [ ]:
def preprocessing(img_path):

  ''' CROPPING '''
  img_cropped, width_original, height_original, x_crop, y_crop = crop_image(img_path)

  ''' RESIZE '''
  img_resize, width_pre_res, height_pre_res = resize_image(img_cropped, (256,256))

  ''' SPECULAR HIGHLIGHTS REMOVE '''
  img_specH = remove_specular_highlights(img_resize)

  ''' GAMMA CORRECTION '''
  img_gamma_corrected = gamma_correction(img_specH)

  output_image = Image.fromarray(img_gamma_corrected.astype(np.uint8))

  return output_image, width_original, height_original, x_crop, y_crop, width_pre_res, height_pre_res

# Postprocessing

In [ ]:
# funzioni di postprocessing

''' FILL HOLES '''

def apply_binary_fill_holes(mask):
    # Applica il fill sui buchi nella maschera binaria
    filled_mask = binary_fill_holes(mask).astype(np.uint8)
    return filled_mask


''' FILTRAGGIO AREE PICCOLE '''

def filter_small_areas(image, min=2500):
    # Etichetta le regioni connesse
    labeled_image = label(image)
    # Creazione di una nuova maschera filtrata
    filtered_mask = np.zeros_like(image)
    # Itera sulle regioni per filtrare quelle piccole
    for region in regionprops(labeled_image):
        if region.area >= min:
            filtered_mask[labeled_image == region.label] = 1
    return filtered_mask

''' DILATAZIONE '''

def apply_binary_dilation(mask, structure, iterations):
    # Applica dilatazione sulla maschera binaria
    dilated_mask = binary_dilation(mask, structure=structure, iterations=iterations).astype(np.uint8)
    return dilated_mask

In [ ]:
def postprocessing(mask, width_original, height_original, x_crop, y_crop, width_pre_res, height_pre_res):

    ''' FILL HOLES '''
    filled_mask = apply_binary_fill_holes(mask)

    ''' FILTRAGGIO AREE PICCOLE '''
    refined_mask = filter_small_areas(filled_mask, min=350)

    ''' RESIZE (upsampling) '''
    mask_resized = resize_mask(np.array(refined_mask), (height_pre_res, width_pre_res), order=0, anti_aliasing=False)
    assert mask_resized.shape == (height_pre_res, width_pre_res), "Resize mismatch!"

    ''' PADDING (upsampling) '''
    mask_padded = padding(mask_resized, width_original, height_original, x_crop, y_crop)
    assert mask_padded.shape == (height_original, width_original), "Padding mismatch!"

    # binarizzazione
    mask_padded = mask_padded > 0

    ''' DILATAZIONE '''
    dilated_mask = apply_binary_dilation(mask_padded,  structure=disk(3), iterations=2)

    return dilated_mask

# Inference Loop

nostra funzione inferenza

In [ ]:
# Definizione funzione per l'inferenza
def run_inference(phase, img_dir, out_dir, save_debug=False):
    """Lancia inferenza e salva maschere (più opz. softmax/debug)."""
    masks_dir = f"{out_dir}/{phase}/masks"
    soft_dir  = f"{out_dir}/{phase}/softmaxs"
    dbg_dir   = f"{out_dir}/{phase}/debugs"
    os.makedirs(masks_dir, exist_ok=True)
    os.makedirs(soft_dir,  exist_ok=True)
    if save_debug:
        os.makedirs(dbg_dir, exist_ok=True)

    for name in tqdm(sorted(os.listdir(img_dir)), desc=phase):
        tile = mmcv.imread(f"{img_dir}/{name}")
        with torch.no_grad():
            res   = inference_model(model, tile)
        logit = res.seg_logits.data.cpu()
        soft  = torch.softmax(logit, dim=0).numpy().astype("float16")

        mask  = soft.argmax(axis=0).astype("uint8")
        mask  = np.where(mask > 1, 255, mask)            # mappa classi >1 a 255
        Image.fromarray(mask).save(f"{masks_dir}/{name}")

        # np.savez_compressed(f"{soft_dir}/{name.replace('.png','.npz')}", soft)  # se serve

        if save_debug:
            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
            ax1.imshow(cv2.cvtColor(tile, cv2.COLOR_BGR2RGB)); ax1.axis("off")
            ax2.imshow(mask, cmap="gray"); ax2.axis("off")
            plt.tight_layout(); plt.savefig(f"{dbg_dir}/{name.replace('.png','_dbg.png')}"); plt.close()

        del res, logit, soft, mask; gc.collect(); torch.cuda.empty_cache()


run_inference("test", f"{DATA}/test/images_filtered_2", OUT, save_debug=True) #cambio save debug True ----------

Loop Inferenza per ciascuna immagine:

1.   Preprocessing
2.   Inferenza
3.   Postprocessing

In [ ]:
# NB
#
# Here you can change everything, just make sure that you write one mask for each image of the subject in the correct format.
# You need to create one folder for each subject and save the predicted masks in that folder.
#
# Tips:
#       - add some checks on the number of items you create, make sure you have one folder for each subject and all the masks for that subject have been written
#       - images have different shapes, some models might require dimensions being multiple of 16 or 32, use padding and resizing with caution. Make sure to save the mask in the original dimensions.
#       - add checks to make sure masks are in the 0-255 format and not 0-1


###############################
####### Inference Loop ########
###############################

# Loop over the subjects and their images
for subject, img_list in subject_dict.items():
    # Create a folder for each subject in the results folder
    subject_folder = os.path.join(results_folder, subject)
    if not os.path.exists(subject_folder):
        os.makedirs(subject_folder)

    ### NB: if you want to apply preprocessing on the 3D volume you need to reconstruct
    # it here and change the loop to cycle over the slices of the volume instead of
    # cycling over the images

    for img_name in tqdm(img_list, desc=f'Processing {subject}'):
        if img_name.endswith('.png'):
            img_path = os.path.join(test_img_folder, img_name)

            ###############################
            ##### 2D Pre-Processing #######
            ###############################

            img = Image.open(img_path)
            img = np.array(img)

            # ...

            # Convert back to PIL Image if necessary for the model
            img = Image.fromarray((img * 255).astype(np.uint8), mode='L')

            # Save img to temporary file
            img.save('tmp_img.png')


            ###############################
            #######    Inference   ########
            ###############################

            result = inference_model(model, 'tmp_img.png')

            # Get data from the result
            pred_label = result.pred_sem_seg.data.squeeze()
            pred_label = pred_label.cpu().numpy().astype(np.uint8)

            ###############################
            ###### 2D Post-Processing #####
            ###############################

            # ...

            ### NB: if you want to apply postprocessing on the 3D volume you need to first create
            # the segmentation volume then apply the post-processing.
            # Finally, using another loop to save the masks

            ###############################
            #######   Save results  #######
            ###############################

            # Save the result
            pred_label_img = Image.fromarray(pred_label, mode='L')
            pred_label_img.save(os.path.join(subject_folder, img_name))


In [ ]:
# Loop attraverso le immagini di una sequenza

    # Dizionario per memorizzare i risultati per ciascuna immagine in una sequenza
    results = {}

    for img_name in tqdm(img_list, desc=f'Processing and evaluating sequence {sequence}'):
        if img_name.endswith('.jpg'):
            img_path = os.path.join(test_img_folder, sequence, f'images_seq{sequence_number}',  img_name)
            if not os.path.exists(img_path):
                print(img_path)

            ###############################
            ##### 2D Pre-Processing #######
            ###############################

            img_prepocessed, width_original, height_original, x_crop, y_crop, width_pre_res, height_pre_res = preprocessing(img_path)

            # Save img to temporary file
            img_prepocessed.save('tmp_img.png')

            ###############################
            #######    Inference   ########
            ###############################

            result = inference_model(model, 'tmp_img.png')

            # Get data from the result
            pred_label = result.pred_sem_seg.data.squeeze()
            pred_label = pred_label.cpu().numpy().astype(np.uint8)

            ###############################
            ###### 2D Post-Processing #####
            ###############################

            pred_label_post = postprocessing(pred_label, width_original, height_original, x_crop, y_crop, width_pre_res, height_pre_res)

            ###############################
            #######   Salvataggio   #######
            ###############################

            # Salva la maschera predetta nella cartella di destinazione
            os.makedirs(destination_folder, exist_ok=True)
            pred_mask_path = os.path.join(destination_folder, img_name)
            Image.fromarray(pred_label_post.astype(np.uint8) * 255).save(pred_mask_path)

Processing and evaluating sequence seq23: 100%|██████████| 56/56 [01:06<00:00,  1.18s/it]


__________

Metriche

In [ ]:
GROUND_TRUTH_PATH = "/content/drive/MyDrive/Colab Notebooks/PolypGen-EIM-24-25/Result_unet_senzapesi_con_preprocessing_1/test/masks/"
PREDICTED_PATH = "/content/drive/MyDrive/Colab Notebooks/PolypGen-EIM-24-25/sequenceData/seq2/masks_seq2/"
# Umbral para considerar un píxel como parte de la máscara (para binarización)
MASK_THRESHOLD = 127 # Para imágenes uint8, 127 es un buen umbral para binario (0 o 255)
DETECTION_THRESHOLD_DICE = 0.5 # Umbral de Dice para considerar un polipo como "detectado" para persistencia

In [ ]:
# --- 2. FUNZIONI DI SUPPORTO GENERALI ---

def load_mask(filepath, debug=False):
    """
    Carica un'immagine di maschera, la converte in scala di grigi se necessario,
    e la prepara assumendo che i pixel siano già 0 o 1.
    Aggiunto 'debug=True' per il debug dei valori e la visualizzazione.
    """
    mask = cv2.imread(filepath, cv2.IMREAD_UNCHANGED) # Carica l'immagine così com'è
    if mask is None:
        raise FileNotFoundError(f"Impossibile caricare l'immagine: {filepath}")

    # Se l'immagine ha 3 o 4 canali (RGB o RGBA), convertire in scala di grigi
    if len(mask.shape) > 2:
        mask = cv2.cvtColor(mask, cv2.COLOR_BGR2GRAY)

    # Assicurarsi che il tipo di dato sia uint8
    mask = mask.astype(np.uint8)

    if debug:
        print(f"\n--- Debugging maschera: {os.path.basename(filepath)} ---")
        print(f"Dimensioni originali: {mask.shape}, Tipo di dato: {mask.dtype}")
        unique_vals = np.unique(mask, return_counts=True)
        print(f"Valori di pixel unici e i loro conteggi (maschera originale): {unique_vals}")

    # Dato che sappiamo che le maschere contengono solo 0 e 1,
    # semplicemente convertiamo i valori diversi da zero (che sono 1) in 1,
    # e 0 rimane 0. Questo è il modo più diretto per garantire il formato 0/1.
    binary_mask = (mask > 0).astype(np.uint8)

    if debug:
        print(f"Valori di pixel unici (dopo la conversione a 0 o 1): {np.unique(binary_mask)}")
        print(f"Somma totale di pixel '1' (polipo) nella maschera binarizzata: {np.sum(binary_mask)}")
        try: # Blocco try-except specifico per la visualizzazione di plt
            plt.figure(figsize=(4,4))
            plt.imshow(binary_mask, cmap='gray')
            plt.title(f"Binarizzata: {os.path.basename(filepath)}")
            plt.axis('off')
            plt.show()
        except Exception as e:
            print(f"Attenzione: Impossibile visualizzare la maschera per {os.path.basename(filepath)} - Errore plt: {e}")

    return binary_mask

def calculate_tp_fp_fn(gt_mask, pred_mask):
    """Calcola Veri Positivi (TP), Falsi Positivi (FP), Falsi Negativi (FN)."""
    gt_mask_bool = gt_mask.astype(bool)
    pred_mask_bool = pred_mask.astype(bool)

    TP = np.sum(gt_mask_bool & pred_mask_bool)
    FP = np.sum(~gt_mask_bool & pred_mask_bool)
    FN = np.sum(gt_mask_bool & ~pred_mask_bool)
    TN = np.sum(~gt_mask_bool & ~pred_mask_bool)

    return TP, FP, FN, TN

# --- 3. METRICHE A LIVELLO DI OGNI FRAME ---

def calculate_dice(gt_mask, pred_mask):
    """Calcola il Coefficiente di Dice."""
    TP, FP, FN, _ = calculate_tp_fp_fn(gt_mask, pred_mask)

    # Caso speciale: oggetto assente in GT ma presente in Pred -> Dice = 0
    if np.sum(gt_mask) == 0 and np.sum(pred_mask) > 0:
        return 0.0
    # Evitare divisione per zero se entrambe le maschere sono vuote (entrambe sono fondo)
    elif (np.sum(gt_mask) == 0 and np.sum(pred_mask) == 0):
        return 1.0
    else:
        return (2.0 * TP) / (2.0 * TP + FP + FN)

def calculate_iou(gt_mask, pred_mask):
    """Calcola Intersection over Union (IoU)."""
    TP, FP, FN, _ = calculate_tp_fp_fn(gt_mask, pred_mask)

    if (TP + FP + FN) == 0:
        return 1.0
    else:
        return TP / (TP + FP + FN)

def calculate_precision_recall_f1(gt_mask, pred_mask):
    """Calcola Precisione, Richiamo e F1-Score."""
    TP, FP, FN, TN = calculate_tp_fp_fn(gt_mask, pred_mask)

    precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
    recall = TP / (TP + FN) if (TP + FN) > 0 else 0.0

    accuracy = (TP + TN) / (TP + FP + FN + TN) if (TP + FP + FN + TN) > 0 else 1.0

    if (precision + recall) == 0:
        f1_score = 0.0
    else:
        f1_score = 2 * (precision * recall) / (precision + recall)

    return precision, recall, f1_score, accuracy

def get_mask_centroid(mask):
    """Calcola il centroide (x, y) di una maschera binaria. Restituisce None se la maschera è vuota."""
    # Moltiplicare per 255 è necessario perché findContours si aspetta 0 o 255.
    mask_uint8 = mask.astype(np.uint8) * 255
    contours, _ = cv2.findContours(mask_uint8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if not contours:
        return None

    largest_contour = max(contours, key=cv2.contourArea)

    M = cv2.moments(largest_contour)
    if M["m00"] == 0:
        return None

    cX = int(M["m10"] / M["m00"])
    cY = int(M["m01"] / M["m00"])
    return (cX, cY)

# --- 4. CALCOLO DELLE METRICHE PER LA SEQUENZA COMPLETA ---

def calculate_sequence_metrics(gt_masks, pred_masks):
    """
    Calcola tutte le metriche per la sequenza completa.
    Argomenti:
        gt_masks (list): Lista di maschere ground truth (array NumPy binarizzati).
        pred_masks (list): Lista di maschere predette (array NumPy binarizzati).
    Restituisce:
        dict: Dizionario con tutte le metriche calcolate.
    """
    if not gt_masks or not pred_masks or len(gt_masks) != len(pred_masks):
        raise ValueError("Le liste di maschere GT e predette devono essere non vuote e avere la stessa lunghezza.")

    num_frames = len(gt_masks)

    dice_scores = []
    iou_scores = []
    precision_scores = []
    recall_scores = []
    f1_scores = []
    accuracy_scores = []

    gt_areas = [] # Nuova lista per memorizzare l'area del ground truth
    pred_areas = [] # Nuova lista per memorizzare l'area predetta

    pred_centroids = []

    frame_detected_dice = []

    print(f"Calcolo metriche per {num_frames} frame...")

    for i in range(num_frames):
        gt = gt_masks[i]
        pred = pred_masks[i]

        dice = calculate_dice(gt, pred)
        iou = calculate_iou(gt, pred)
        precision, recall, f1, accuracy = calculate_precision_recall_f1(gt, pred)

        dice_scores.append(dice)
        iou_scores.append(iou)
        precision_scores.append(precision)
        recall_scores.append(recall)
        f1_scores.append(f1)
        accuracy_scores.append(accuracy)

        gt_areas.append(np.sum(gt)) # Aggiunge l'area del ground truth
        pred_areas.append(np.sum(pred)) # Aggiunge l'area predetta

        pred_c = get_mask_centroid(pred)
        pred_centroids.append(pred_c)

        frame_detected_dice.append(1 if dice >= DETECTION_THRESHOLD_DICE else 0)

    results = {
        "Dice": {"mean": np.mean(dice_scores), "std": np.std(dice_scores)},
        "IoU": {"mean": np.mean(iou_scores), "std": np.std(iou_scores)},
        "Precision": {"mean": np.mean(precision_scores), "std": np.std(precision_scores)},
        "Recall": {"mean": np.mean(recall_scores), "std": np.std(recall_scores)},
        "F1-Score": {"mean": np.mean(f1_scores), "std": np.std(f1_scores)},
        "Accuracy": {"mean": np.mean(accuracy_scores), "std": np.std(accuracy_scores)},
        "Area GT (pixel)": {"mean": np.mean(gt_areas), "std": np.std(gt_areas)}, # Nuova metrica
        "Area Predetta (pixel)": {"mean": np.mean(pred_areas), "std": np.std(pred_areas)}, # Nuova metrica
    }

    dice_fluctuations = [abs(dice_scores[i+1] - dice_scores[i]) for i in range(num_frames - 1)]
    results["Fluttuazione Temporale del Dice"] = {
        "mean": np.mean(dice_fluctuations) if dice_fluctuations else 0.0,
        "std": np.std(dice_fluctuations) if dice_fluctuations else 0.0
    }

    pred_centroid_distances = []
    for i in range(num_frames - 1):
        c1 = pred_centroids[i]
        c2 = pred_centroids[i+1]

        if c1 is not None and c2 is not None:
            dist = euclidean(c1, c2)
            pred_centroid_distances.append(dist)

    results["Jitter del Centroide Maschera Predetta"] = {
        "mean": np.mean(pred_centroid_distances) if pred_centroid_distances else 0.0,
        "std": np.std(pred_centroid_distances) if pred_centroid_distances else 0.0
    }

    detection_streaks = []
    current_streak = 0
    for detected in frame_detected_dice:
        if detected == 1:
            current_streak += 1
        else:
            if current_streak > 0:
                detection_streaks.append(current_streak)
            current_streak = 0
    if current_streak > 0:
        detection_streaks.append(current_streak)

    results["Persistenza della Rilevazione (Media Strisce)"] = {
        "mean": np.mean(detection_streaks) if detection_streaks else 0.0,
        "std": np.std(detection_streaks) if detection_streaks else 0.0
    }
    results["Totale Frame Rilevati (>Dice Soglia)"] = np.sum(frame_detected_dice)
    results["Totale Frame"] = num_frames


    return results


In [ ]:
#
gt_mask_files = sorted(glob(os.path.join(GROUND_TRUTH_PATH, '*.png')))
pred_mask_files = sorted(glob(os.path.join(PREDICTED_PATH, '*.png')))

if not gt_mask_files or not pred_mask_files:
    print("Errore: Nessun file di maschera trovato nei percorsi specificati.")
    print(f"Percorso GT: {GROUND_TRUTH_PATH}, Trovati: {len(gt_mask_files)} file")
    print(f"Percorso Pred: {PREDICTED_PATH}, Trovati: {len(pred_mask_files)} file")
else:
    gt_filenames = [os.path.basename(f) for f in gt_mask_files]
    pred_filenames = [os.path.basename(f) for f in pred_mask_files]

    common_filenames = sorted(list(set(gt_filenames) & set(pred_filenames)))

    if not common_filenames:
        print("Errore: Nessun file con nomi corrispondenti trovato in entrambe le cartelle.")
    else:
        gt_masks_data = []
        pred_masks_data = []

        for i, filename in enumerate(common_filenames):
            gt_path = os.path.join(GROUND_TRUTH_PATH, filename)
            pred_path = os.path.join(PREDICTED_PATH, filename)

            # Attiva la modalità debug per i primi 5 frame
            debug_this_frame = (i < 5)

            try:
                gt_masks_data.append(load_mask(gt_path, debug=debug_this_frame))
                pred_masks_data.append(load_mask(pred_path, debug=debug_this_frame))
            except FileNotFoundError as e:
                print(e)
                continue
            except Exception as e:
                print(f"Errore durante l'elaborazione di {filename}: {e}")
                continue

        if gt_masks_data and pred_masks_data:
            print(f"\nElaborazione di {len(gt_masks_data)} coppie di maschere...")
            all_metrics = calculate_sequence_metrics(gt_masks_data, pred_masks_data)

            print("\n--- RISULTATI DELLE METRICHE DI VALUTAZIONE ---")
            print("\nMetriche per Frame (Media ± Deviazione Standard):")
            for metric_name, values in all_metrics.items():
                if isinstance(values, dict) and "mean" in values and "std" in values and metric_name not in ["Fluttuazione Temporale del Dice", "Jitter del Centroide Maschera Predetta", "Persistenza della Rilevazione (Media Strisce)"]:
                    print(f"- {metric_name}: {values['mean']:.4f} ± {values['std']:.4f}")

            print("\nMetriche di Coerenza Temporale e Persistenza:")
            if "Fluttuazione Temporale del Dice" in all_metrics:
                val = all_metrics["Fluttuazione Temporale del Dice"]
                print(f"- Fluttuazione Temporale del Dice: {val['mean']:.4f} ± {val['std']:.4f} (Un valore più basso indica maggiore stabilità)")

            if "Jitter del Centroide Maschera Predetta" in all_metrics:
                val = all_metrics["Jitter del Centroide Maschera Predetta"]
                print(f"- Jitter del Centroide Maschera Predetta: {val['mean']:.2f} ± {val['std']:.2f} pixel (Un valore più basso indica minore tremolio)")

            if "Persistenza della Rilevazione (Media Strisce)" in all_metrics:
                val = all_metrics["Persistenza della Rilevazione (Media Strisce)"]
                print(f"- Persistenza della Rilevazione (Media Strisce): {val['mean']:.2f} ± {val['std']:.2f} frame")
                print(f"- Totale Frame Rilevati (>Dice {DETECTION_THRESHOLD_DICE}): {all_metrics['Totale Frame Rilevati (>Dice Soglia)']} / {all_metrics['Totale Frame']}")

        else:
            print("Impossibile caricare coppie di maschere valide per l'elaborazione.")